In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Set before importing torch. This helps reduce CUDA allocator fragmentation on Colab.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

os.makedirs('/content/drive/MyDrive/mamba-vs-lstm/results', exist_ok=True)
print('Drive mounted')


In [ ]:
# Colab already includes CUDA-enabled PyTorch. Avoid reinstalling torch in a T4 runtime.
!pip install pandas scikit-learn matplotlib seaborn -q


In [ ]:
# This T4-safe version uses the Mamba-style forecasting block defined below.
# No external CUDA extension is required, which keeps it friendly to Colab T4 runtimes.
print('Using notebook-defined T4-safe Mamba forecaster')


In [ ]:
from google.colab import files
uploaded = files.upload()   # upload jena_climate_2009_2016.csv
!ls jena_climate_2009_2016.csv
print("✅ Dataset uploaded")

In [ ]:
import sys
sys.path.insert(0, '/content')

import gc
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**2
else:
    total_vram = 0

print('All imports done')
print(f'Device     : {DEVICE}')
print(f'VRAM Total : {total_vram:.0f} MB')
print(f'PyTorch    : {torch.__version__}')


In [ ]:
def load_data(path):
    df = pd.read_csv(path)
    features = ['T (degC)', 'p (mbar)', 'rh (%)']
    df = df[features]
    df.replace(-9999.0, np.nan, inplace=True)
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'Loaded {len(df)} rows | Columns: {list(df.columns)}')
    return df


def normalize(df):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df.values).astype(np.float32, copy=False)
    return scaled, scaler


def count_windows(data, input_steps, output_steps):
    return max(0, len(data) - input_steps - output_steps + 1)


class ClimateWindowDataset(Dataset):
    """Lazy sliding-window dataset.

    This avoids materializing the entire X tensor, which is the main RAM spike in the
    original notebook. Each sample is sliced only when the DataLoader asks for it.
    """

    def __init__(self, data, start_idx, end_idx, input_steps, output_steps):
        self.data = data
        self.start_idx = int(start_idx)
        self.end_idx = int(end_idx)
        self.input_steps = int(input_steps)
        self.output_steps = int(output_steps)

    def __len__(self):
        return max(0, self.end_idx - self.start_idx)

    def __getitem__(self, idx):
        i = self.start_idx + idx
        x = self.data[i : i + self.input_steps]
        y = self.data[i + self.input_steps : i + self.input_steps + self.output_steps, 0]
        return torch.from_numpy(x), torch.from_numpy(y.copy())


print('Data utils defined')


In [ ]:
INPUT_STEPS = 168
OUTPUT_STEPS = 24
BATCH_SIZE = 64          # microbatch size that fits comfortably on a T4
EVAL_BATCH_SIZE = 128    # eval has no backward pass, so it can be a little larger
GRAD_ACCUM_STEPS = 8     # effective train batch = 64 * 8 = 512
NUM_WORKERS = min(2, os.cpu_count() or 0)


df = load_data('jena_climate_2009_2016.csv')
scaled, scaler = normalize(df)

n_windows = count_windows(scaled, INPUT_STEPS, OUTPUT_STEPS)
t = int(n_windows * 0.7)
v = int(n_windows * 0.85)

train_ds = ClimateWindowDataset(scaled, 0, t, INPUT_STEPS, OUTPUT_STEPS)
val_ds   = ClimateWindowDataset(scaled, t, v, INPUT_STEPS, OUTPUT_STEPS)
test_ds  = ClimateWindowDataset(scaled, v, n_windows, INPUT_STEPS, OUTPUT_STEPS)

loader_common = dict(
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == 'cuda'),
)
if NUM_WORKERS > 0:
    loader_common.update(prefetch_factor=2, persistent_workers=True)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_common)
val_dl   = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, **loader_common)
test_dl  = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, **loader_common)

print('DataLoaders ready')
print(f'   Train : {len(train_ds)} samples | {len(train_dl)} microbatches')
print(f'   Val   : {len(val_ds)} samples   | {len(val_dl)} batches')
print(f'   Test  : {len(test_ds)} samples  | {len(test_dl)} batches')
print(f'   Effective train batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}')


In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=3, hidden_size=128,
                 num_layers=2, output_steps=24, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden_size, output_steps)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

print("✅ LSTMForecaster defined")

In [ ]:
from torch.utils.checkpoint import checkpoint

# T4-safe Mamba-style config. If peak VRAM stays well below 12 GB, try D_MODEL=384.
D_MODEL = 256
N_LAYERS = 3
EXPAND = 2
D_CONV = 3
DROPOUT = 0.1
USE_GRAD_CHECKPOINTING = True


class SSMBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, expand=EXPAND, d_conv=D_CONV, dropout=DROPOUT):
        super().__init__()
        d_inner = d_model * expand
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            d_inner,
            d_inner,
            kernel_size=d_conv,
            padding=d_conv - 1,
            groups=d_inner,
            bias=True,
        )
        self.dt_proj = nn.Linear(d_inner, d_inner, bias=True)
        self.D = nn.Parameter(torch.ones(d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        seq_len = x.shape[1]
        x = self.norm(x)
        x1, z = self.in_proj(x).chunk(2, dim=-1)

        # Causal depthwise conv. Crop back to seq_len so residual/gate shapes match.
        x1 = self.conv1d(x1.transpose(1, 2))[:, :, :seq_len].transpose(1, 2)
        x1 = F.silu(x1)

        dt = F.softplus(self.dt_proj(x1))
        y = (x1 * self.D + dt * x1) * F.silu(z)
        return residual + self.dropout(self.out_proj(y))


class MambaForecaster(nn.Module):
    def __init__(self, input_size=3, d_model=D_MODEL, n_layers=N_LAYERS, output_steps=OUTPUT_STEPS):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.blocks = nn.ModuleList([SSMBlock(d_model) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, output_steps)

    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            if self.training and USE_GRAD_CHECKPOINTING:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        x = self.norm(x)
        return self.fc(x[:, -1, :])


def build_mamba():
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    model = MambaForecaster().to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    allocated = torch.cuda.memory_allocated() / 1024**2 if DEVICE == 'cuda' else 0
    print('MambaForecaster built')
    print(f'   Parameters       : {total_params/1e6:.1f}M')
    print(f'   VRAM after model : {allocated:.0f}MB / {total_vram:.0f}MB')
    return model


print('MambaForecaster class defined')
print(f'   Config: d_model={D_MODEL}, layers={N_LAYERS}, microbatch={BATCH_SIZE}, accum={GRAD_ACCUM_STEPS}')


In [ ]:
criterion = nn.MSELoss()
AMP_ENABLED = (DEVICE == 'cuda')


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_seen = 0
    with torch.inference_mode():
        for X, y in loader:
            X = X.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE, enabled=AMP_ENABLED):
                pred = model(X)
                loss = criterion(pred, y)
            total_loss += loss.item() * X.size(0)
            total_seen += X.size(0)
    return total_loss / max(total_seen, 1)


print('criterion + evaluate defined')


In [ ]:
lstm = LSTMForecaster().to(DEVICE)
lstm.load_state_dict(torch.load(
    '/content/drive/MyDrive/mamba-vs-lstm/results/lstm_best.pt',
    map_location=DEVICE
))

lstm_test_mse = evaluate(lstm, test_dl)
print('LSTM loaded from Drive')
print(f'LSTM Test MSE : {lstm_test_mse:.6f}  <- score to beat')

# Free VRAM before constructing/training Mamba.
del lstm
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()


In [ ]:
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

mamba = build_mamba()
optimizer = torch.optim.AdamW(mamba.parameters(), lr=3e-4, weight_decay=1e-4)
scaler_amp = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
EPOCHS = 20
best_val = float('inf')
epoch_times = []
train_losses = []
val_losses = []

print(f'Starting Mamba Training on {DEVICE}')
print('=' * 75)
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'Time':>8} {'PeakVRAM':>10} {'Status':>8}")
print('=' * 75)

total_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    mamba.train()
    train_loss = 0.0
    train_seen = 0
    batch_count = len(train_dl)
    epoch_start = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)
    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    for batch_idx, (X, y) in enumerate(train_dl, 1):
        X = X.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        try:
            with torch.amp.autocast(device_type=DEVICE, enabled=AMP_ENABLED):
                loss = criterion(mamba(X), y)
                scaled_loss = loss / GRAD_ACCUM_STEPS

            scaler_amp.scale(scaled_loss).backward()

            if batch_idx % GRAD_ACCUM_STEPS == 0 or batch_idx == batch_count:
                scaler_amp.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(mamba.parameters(), max_norm=1.0)
                scaler_amp.step(optimizer)
                scaler_amp.update()
                optimizer.zero_grad(set_to_none=True)

            train_loss += loss.item() * X.size(0)
            train_seen += X.size(0)

        except torch.cuda.OutOfMemoryError as exc:
            optimizer.zero_grad(set_to_none=True)
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            raise RuntimeError(
                'CUDA OOM even with the T4-safe settings. Lower BATCH_SIZE to 32 '
                'or D_MODEL to 192 and rerun from the data/model cells.'
            ) from exc

        if batch_idx % 25 == 0 or batch_idx == batch_count:
            pct = batch_idx / batch_count * 100
            done = int(pct / 5)
            bar = '#' * done + '-' * (20 - done)
            vram = torch.cuda.memory_allocated() / 1024**2 if DEVICE == 'cuda' else 0
            avg_loss = train_loss / max(train_seen, 1)
            print(f"\r  [{bar}] {batch_idx}/{batch_count} "
                  f"| Loss: {avg_loss:.6f} "
                  f"| VRAM: {vram:.0f}MB      ",
                  end='', flush=True)

    epoch_time = time.perf_counter() - epoch_start
    train_loss = train_loss / max(train_seen, 1)
    epoch_times.append(epoch_time)
    train_losses.append(train_loss)

    val_loss = evaluate(mamba, val_dl)
    val_losses.append(val_loss)

    peak_vram = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE == 'cuda' else 0

    tag = ''
    if val_loss < best_val:
        best_val = val_loss
        torch.save(mamba.state_dict(), '/content/drive/MyDrive/mamba-vs-lstm/results/mamba_best.pt')
        tag = 'saved'

    elapsed = time.perf_counter() - total_start
    remaining = (elapsed / epoch) * (EPOCHS - epoch)
    eta = f'{int(remaining//60)}m {int(remaining%60)}s'
    free_vram = total_vram - peak_vram if DEVICE == 'cuda' else 0

    print(f"\r{'-' * 75}")
    print(f'  {epoch:02d}/{EPOCHS}'
          f'  {train_loss:>12.6f}'
          f'  {val_loss:>12.6f}'
          f'  {epoch_time:>7.1f}s'
          f'  {peak_vram:>8.0f}MB'
          f'  {tag:>8}')
    print(f'  Elapsed: {int(elapsed//60)}m {int(elapsed%60)}s'
          f'  |  ETA: {eta}'
          f'  |  Best Val: {best_val:.6f}'
          f'  |  VRAM Free: {free_vram:.0f}MB')

print(f"\n{'=' * 75}")
total_time = time.perf_counter() - total_start
print(f'Done in {int(total_time//60)}m {int(total_time%60)}s')

mamba.load_state_dict(torch.load('/content/drive/MyDrive/mamba-vs-lstm/results/mamba_best.pt', map_location=DEVICE))
mamba_test_mse = evaluate(mamba, test_dl)
print(f'Mamba Test MSE  : {mamba_test_mse:.6f}')
print(f'Avg Epoch Time  : {sum(epoch_times)/len(epoch_times):.1f}s')


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display


RESULTS_DIR = Path("/content/drive/MyDrive/mamba-vs-lstm/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_TRAIN_LOSSES = [
    0.004609, 0.000761, 0.000722, 0.000705, 0.000701,
    0.000687, 0.000690, 0.000680, 0.000676, 0.000672,
    0.000667, 0.000665, 0.000662, 0.000660, 0.000650,
    0.000650, 0.000645, 0.000645, 0.000642, 0.000658,
]
DEFAULT_VAL_LOSSES = [
    0.000728, 0.000704, 0.000705, 0.000750, 0.000686,
    0.000762, 0.000713, 0.000673, 0.000733, 0.000677,
    0.000687, 0.000685, 0.000677, 0.000659, 0.000690,
    0.000647, 0.000645, 0.000657, 0.000648, 0.000656,
]

if "lstm_test_mse" not in globals():
    lstm_test_mse = 0.000628
if "mamba_test_mse" not in globals():
    mamba_test_mse = 0.000587
if "train_losses" not in globals():
    train_losses = DEFAULT_TRAIN_LOSSES
if "val_losses" not in globals():
    val_losses = DEFAULT_VAL_LOSSES


summary = pd.DataFrame(
    [
        {"model": "LSTM", "test_mse": float(lstm_test_mse)},
        {"model": "Mamba-style", "test_mse": float(mamba_test_mse)},
    ]
)
summary["relative_to_lstm"] = summary["test_mse"] / float(lstm_test_mse)
summary.to_csv(RESULTS_DIR / "summary_metrics.csv", index=False)
display(summary)


plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker="o", label="Train")
plt.plot(range(1, len(val_losses) + 1), val_losses, marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Mamba Training Curve")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "loss_curve.png", dpi=160)
plt.show()


plt.figure(figsize=(6, 4.5))
bars = plt.bar(summary["model"], summary["test_mse"], color=["#4c78a8", "#f58518"])
plt.ylabel("Test MSE")
plt.title("Normalized Test Error")
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.6f}",
        ha="center",
        va="bottom",
    )
plt.tight_layout()
plt.savefig(RESULTS_DIR / "test_mse_comparison.png", dpi=160)
plt.show()


def inverse_temperature(values):
    values = np.asarray(values, dtype=np.float32)
    flat = values.reshape(-1)
    padded = np.zeros((flat.shape[0], 3), dtype=np.float32)
    padded[:, 0] = flat
    return scaler.inverse_transform(padded)[:, 0].reshape(values.shape)


if all(name in globals() for name in ["mamba", "test_dl", "scaler", "DEVICE"]):
    mamba.eval()
    X_batch, y_batch = next(iter(test_dl))
    with torch.inference_mode():
        with torch.amp.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
            pred_batch = mamba(X_batch.to(DEVICE, non_blocking=True)).float().cpu().numpy()

    actual_norm = y_batch.numpy()
    pred_norm = pred_batch
    actual_deg_c = inverse_temperature(actual_norm[0])
    pred_deg_c = inverse_temperature(pred_norm[0])

    plt.figure(figsize=(8, 4.5))
    plt.plot(actual_deg_c, marker="o", label="Actual")
    plt.plot(pred_deg_c, marker="o", label="Mamba prediction")
    plt.xlabel("Forecast step")
    plt.ylabel("Temperature (degC)")
    plt.title("Sample Test Forecast")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "sample_forecast.png", dpi=160)
    plt.show()
else:
    print("Skipped sample_forecast.png because mamba/test_dl/scaler are not in memory.")
    print("Run this cell immediately after the training cell to generate the forecast plot.")


print(f"Saved result artifacts to: {RESULTS_DIR}")
print(f"LSTM Test MSE   : {lstm_test_mse:.6f}")
print(f"Mamba Test MSE  : {mamba_test_mse:.6f}")
print(f"Improvement     : {(1 - mamba_test_mse / lstm_test_mse) * 100:.2f}% lower MSE")
